# location-maison-model-for-google-collab

Notebook Colab pour lancer le dataset, l'entraînement et l'évaluation **depuis Google Drive**.

Convention utilisée ici :
- le projet synchronisé vit dans `.../location-maison-model-for-google-collab/code`
- les runs se font depuis ce dossier `code/`


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
PROJECT_DRIVE_ROOT = '/content/drive/MyDrive/IA/location-maison-model-for-google-collab'
PROJECT_CODE_DIR = f'{PROJECT_DRIVE_ROOT}/code'
print('PROJECT_DRIVE_ROOT =', PROJECT_DRIVE_ROOT)
print('PROJECT_CODE_DIR  =', PROJECT_CODE_DIR)

# Adapte uniquement cette cellule si ton dossier Drive a un autre chemin.


In [ ]:
%cd {PROJECT_CODE_DIR}
!pwd
!ls


In [ ]:
import torch
print('CUDA dispo :', torch.cuda.is_available())
print('GPU :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'aucun')


In [ ]:
!pip install -r requirements.txt


## Optionnel : token Hugging Face

Si tu veux éviter les limitations de débit, renseigne ton token HF ici. Sinon laisse vide.


In [ ]:
import os
HF_TOKEN = ''
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    os.environ['HUGGINGFACE_HUB_TOKEN'] = HF_TOKEN
    print('HF_TOKEN configured')
else:
    print('HF_TOKEN not set')


## Préparer les sources et générer le dataset

Relance cette cellule si tu modifies les sources (`post-for-facebook`, exports `properties`, annonces archivées).


In [ ]:
!python scripts/dataset/prepare_sources.py
!PYTHONPATH=src python -m location_maison_model_annonce.cli.generate_dataset --config config/project.yaml


In [ ]:
!cat data/datasets/dataset_manifest.json


## Entraînement complet

Cette commande lance le train complet puis l'évaluation métier finale.


In [ ]:
!PYTHONPATH=src python -m location_maison_model_annonce.cli.train --config config/project.yaml


## Évaluation seule

Utile si le train est déjà fini et que tu veux relancer uniquement l'évaluation.


In [ ]:
!PYTHONPATH=src python -m location_maison_model_annonce.cli.evaluate --config config/project.yaml --split test


## Consulter les résultats


In [ ]:
!cat outputs/metrics/latest_metrics.json


In [ ]:
!ls outputs/checkpoints
!ls outputs/reports


## Tester le modèle


In [ ]:
!PYTHONPATH=src python -m location_maison_model_annonce.cli.predict --config config/project.yaml --max-new-tokens 320 --text "Studio à louer 2 chambres douche cuisine wc buanderie dans la barrière avec la grille de sécurité avec l'eau à 120000 f à Okolassi contact 066177172"


## Logs utiles

Lance ces commandes dans une cellule à part quand tu veux inspecter les fichiers produits.


In [ ]:
!tail -n 100 logs/train/train.log
!tail -n 100 logs/app/application.log
